## setup

In [ ]:
import numpy as np
from pathlib import Path

from qick import *

from qick.asm_v2 import AveragerProgramV2, QickProgramV2
from qick.asm_riscv import RvAveragerProgram, RvProgram
from qick.asm_v2 import QickSweep1D

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING, force=True, format='%(levelname)-8s [%(filename)s:%(lineno)d] %(message)s')

In [ ]:
from qick.riscv_toolchain import CC
assert CC is not None, "riscv64-unknown-elf-clang (or -gcc) not found on PATH"

tproc_soccfg = QickConfig(str(Path("../../firmware/testbench/qick_testbench/soccfg.json").resolve()))
print(tproc_soccfg)

from qick.riscq_config import RvConfig
from qick.riscv import _socmap, DEFAULT_SOC
soc = soccfg = RvConfig(_socmap(DEFAULT_SOC))
print(soccfg)

In [ ]:
BACKENDS = {
    'tProc v2': {'soccfg': tproc_soccfg, 'avg': AveragerProgramV2, 'prog': QickProgramV2, 'gen_ch': 0, 'ro_ch': 0},
    'RISC-Q':   {'soccfg': soccfg,       'avg': RvAveragerProgram, 'prog': RvProgram,     'gen_ch': 1, 'ro_ch': 0},
}

def compile_both(program, cfg=None, **kwargs):
    results = {}
    for name, b in BACKENDS.items():
        try:
            if isinstance(program, type):
                cls = type(program.__name__, (program, b['avg']), {})
                prog = cls(b['soccfg'], reps=1, final_delay=0.5, cfg=dict(cfg, gen_ch=b['gen_ch'], ro_ch=b['ro_ch']), **kwargs)
            else:
                prog = b['prog'](b['soccfg'])
                program(prog)
                prog.end()
                prog.compile()
            results[name] = prog
        except Exception as e:
            results[name] = e
    return results

def same_as_qick(label, results):
    def show(r):
        return "ok" if not isinstance(r, Exception) else "refused (%s)" % str(r).splitlines()[0][:70]
    tproc_ok = not isinstance(results['tProc v2'], Exception)
    rv_ok = not isinstance(results['RISC-Q'], Exception)
    print("%-28s tProc v2: %-12s RISC-Q: %s" % (label, show(results['tProc v2']) if tproc_ok else "refused", show(results['RISC-Q'])))
    if tproc_ok and not rv_ok:
        raise AssertionError("%s: QICK accepts this, RISC-Q refuses it: %s" % (label, results['RISC-Q']))
    if rv_ok and not tproc_ok:
        raise AssertionError("%s: QICK refuses this (%s), RISC-Q accepts it" % (label, results['tProc v2']))

def check_limit(label, program, values, make_cfg=None):
    failures = []
    for val in values:
        try:
            if make_cfg is None:
                same_as_qick("%s %s" % (label, val), compile_both(program(val)))
            else:
                same_as_qick("%s %s" % (label, val), compile_both(program, make_cfg(val)))
        except AssertionError as e:
            failures.append(str(e))
    assert not failures, "\n".join(failures)

### delay and wait
Same idea as LongProgram in 00_intro, pushed out to the limit.

In [ ]:
class DelayProgram:
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']

        self.declare_gen(ch=gen_ch, nqz=1)
        self.declare_readout(ch=ro_ch, length=0.3)

        self.add_pulse(ch=gen_ch, name="mypulse", ro_ch=ro_ch,
                       style="const",
                       freq=599,
                       length=0.05,
                       phase=0,
                       gain=1.0,
                      )

    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="mypulse", t=0)
        # times are in ticks of this machine's timeline
        if 'delay' in cfg:
            self.delay(self.soccfg.cycles2us(cfg['delay']), tag='delay')
        if 'wait' in cfg:
            self.wait(self.soccfg.cycles2us(cfg['wait']), tag='wait')

check_limit("delay", DelayProgram, [2**31 - 1, 2**31], lambda n: {'delay': n})

In [ ]:
check_limit("wait", DelayProgram, [2**31 - 1, 2**31], lambda n: {'wait': n})

In [ ]:
results = compile_both(DelayProgram, {'delay': 2**31 - 1})
for name, prog in results.items():
    t = prog.get_time_param('delay', 't')
    print("%-8s delay = %.6f s = %d ticks" % (name, t/1e6, prog.soccfg.us2cycles(t)))
    assert prog.soccfg.us2cycles(t) == 2**31 - 1

### pulse times
Pulse times are relative to the reference time, and QICK adds a little bit of its own offset, so the largest usable `t` is slightly under 2**31 ticks.

In [ ]:
class PulseTimeProgram(DelayProgram):
    def _body(self, cfg):
        self.trigger(ros=[cfg['ro_ch']], t=self.soccfg.cycles2us(cfg['t']))
        self.pulse(ch=cfg['gen_ch'], name="mypulse", t=self.soccfg.cycles2us(cfg['t']))

check_limit("pulse t", PulseTimeProgram, [2**31 - 2**16, 2**31], lambda n: {'t': n})

## pulses

Pulse length, gain, frequency and phase are all checked (or wrapped) by QICK before the backend sees them, so these should match already.

In [ ]:
class PulseProgram:
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']

        self.declare_gen(ch=gen_ch, nqz=1)
        self.declare_readout(ch=ro_ch, length=0.3)

        self.add_pulse(ch=gen_ch, name="mypulse", ro_ch=ro_ch,
                       style="const",
                       freq=cfg.get('freq', 599),
                       length=self.soccfg.cycles2us(cfg.get('length', 30), gen_ch=gen_ch),
                       phase=cfg.get('phase', 0),
                       gain=cfg.get('gain', 1.0),
                      )

    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="mypulse", t=0)

check_limit("length", PulseProgram, [2, 3, 2**16 - 1, 2**16], lambda n: {'length': n})

In [ ]:
check_limit("gain", PulseProgram, [1.0, -1.0], lambda g: {'gain': g})

In [ ]:
check_limit("freq", PulseProgram, [-3000, 0, 4000, 20000], lambda f: {'freq': f})
check_limit("phase", PulseProgram, [-720, 3600], lambda p: {'phase': p})

## loops and sweeps

### loop counts
The tProc v2 compares the loop counter against `n - 1` as a 24-bit signed literal, so a loop can have at most 2**23 passes.

In [ ]:
class LoopProgram(PulseProgram):
    def _initialize(self, cfg):
        self.add_loop("myloop", cfg['steps'])
        super()._initialize(cfg)

check_limit("loop count", LoopProgram, [2**23, 2**23 + 1, 2**31], lambda n: {'steps': n})

### sweep resolution
QICK refuses a sweep whose step rounds to zero. Gain resolution is the same on both (`maxv` = 32766), so the same sweeps should pass and fail.

In [ ]:
def sweep_cfg(steps):
    return {'steps': steps, 'gain': QickSweep1D("myloop", 0.0, 1.0)}

check_limit("gain sweep, steps", LoopProgram, [32767, 32768], sweep_cfg)

## registers and memory



### register values
`write_reg` takes a signed 32-bit literal. `inc_reg` truncates its argument to 32 bits instead of refusing it, so every value is accepted.

In [ ]:
def write_reg_program(val):
    def program(prog):
        prog.add_reg(name='myreg')
        prog.write_reg(dst='myreg', src=val)
    return program

def inc_reg_program(val):
    def program(prog):
        prog.add_reg(name='myreg')
        prog.inc_reg(dst='myreg', src=val)
    return program

check_limit("write_reg", write_reg_program, [2**31 - 1, 2**31, -2**31, -2**31 - 1])

In [ ]:
check_limit("inc_reg", inc_reg_program, [2**31 - 1, 2**31, -2**31 - 1])

### conditional jumps
`cond_jump` compares against a 24-bit signed literal, and shifts (`>>`) go up to 15.

In [ ]:
def cond_jump_program(op, val):
    def program(prog):
        prog.add_reg(name='myreg')
        prog.write_reg(dst='myreg', src=0)
        prog.label('top')
        prog.cond_jump('top', 'myreg', 'NZ', op=op, arg2=val)
    return program

check_limit("cond_jump -", lambda v: cond_jump_program('-', v), [2**23 - 1, 2**23])

In [ ]:
check_limit("cond_jump >>", lambda v: cond_jump_program('>>', v), [15, 16])

### data memory
A literal data-memory address can go up to 1023. QICK accepting the address isn't enough here: RISC-Q also has to have memory behind it, so we check the size of the data memory in the built image too.

In [ ]:
def dmem_program(addr):
    def program(prog):
        prog.add_reg(name='myreg')
        prog.write_dmem(addr=addr, src=17)
        prog.read_dmem(dst='myreg', addr=addr)
    return program

check_limit("dmem addr", dmem_program, [1023, 1024])

In [ ]:
prog = compile_both(dmem_program(1023))['RISC-Q']
dmem_bytes = prog.image.symbols['__qick_dmem'][1]
print("RISC-Q data memory: %d words" % (dmem_bytes//4))
assert dmem_bytes >= 4*1024, "QICK can address 1024 words of data memory, RISC-Q only has %d" % (dmem_bytes//4)

### data registers
Both configs say 16 data registers. Loops, sweeps and swept times use them up too.

In [ ]:
def many_regs_program(n):
    def program(prog):
        for i in range(n):
            prog.add_reg(name='reg%d' % i)
        prog.write_reg(dst='reg%d' % (n-1), src=1)
    return program

check_limit("data registers", many_regs_program, [16, 17])